# Image preprocessing without repository assets

## Goal

This walkthrough creates a few synthetic RGB and grayscale images inside a temporary directory. AutoPrepML validates, resizes, converts, normalizes, and augments them. The temporary directory is removed before the final check, so no image file is committed or left behind.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
from PIL import Image
from autoprepml import ImagePrepML

with TemporaryDirectory(prefix="autoprepml-images-") as temporary_root:
    image_root = Path(temporary_root)
    Image.new("RGB", (16, 12), color=(40, 90, 140)).save(image_root / "blue.png")
    Image.new("L", (10, 20), color=180).save(image_root / "gray.png")
    Image.new("RGB", (24, 24), color=(200, 80, 40)).save(image_root / "orange.png")

    preparer = ImagePrepML(
        image_dir=str(image_root), target_size=(24, 24),
        color_mode="rgb", normalize=True
    )
    issues = preparer.detect(verbose=False)
    processed = preparer.clean(
        augment=True,
        augmentation_config={"horizontal_flip": True, "rotations": [90], "include_original": True},
    )
    print("input_images:", len(preparer.image_paths))
    print("issue_categories:", sorted(issues))
    print("processed_shape:", processed.shape)
    print("pixel_range:", float(processed.min()), float(processed.max()))

print("temporary_images_deleted: true")

input_images: 3
issue_categories: ['color_mode_issues', 'low_quality', 'size_mismatch']
processed_shape: (9, 24, 24, 3)
pixel_range: 0.1568627506494522 0.7843137383460999
temporary_images_deleted: true


## Checks

In [2]:
assert processed.shape == (9, 24, 24, 3)
assert processed.dtype == np.float32
assert 0.0 <= float(processed.min()) <= float(processed.max()) <= 1.0
assert not image_root.exists()
print("Image workflow checks passed.")

Image workflow checks passed.
